# Adversarial validation

Question: how different is the training domain from the test domain, and which train data looks like test?

The assignment asks for adversarial validation with the same model family as the main task, so the classifier is a BiGRU over word embeddings. The main model is not chosen yet: `RNN` below is the one line to change once it is.

Texts are compared as sentences, because train is sentence-level and test is post-level. Otherwise length alone would separate the domains.

In [1]:
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

DATA = Path("../data")
DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
RNN = nn.GRU
N_TRAIN, MAX_TOKENS, EPOCHS = 30_000, 64, 4
torch.manual_seed(0)
np.random.seed(0)
pd.set_option("display.max_colwidth", 160)

In [2]:
TOKEN = re.compile(r"\w+(?:['’ʼ-]\w+)*|[^\w\s]")
SENTENCE = re.compile(r"(?<=[.!?…])\s+|\n+")
strip_noise = lambda t: re.sub(r"[^\w\s'’ʼ-]|\d", " ", t)

uk = pd.read_csv(DATA / "uk_geo_dataset.csv", usecols=["text", "is_valid"])
uk = uk[~uk["text"].str.contains(r"TCH\.?\s*$")]  # section footers of one news site, a known artefact
test = pd.read_csv(DATA / "test.csv")
test_sents = [x.strip() for t in test["text"] for x in SENTENCE.split(t) if len(x.strip()) >= 15]

ru_parts = [c.sample(frac=0.005, random_state=i) for i, c in enumerate(pd.read_csv(DATA / "ru_geo_dataset.csv", usecols=["text"], chunksize=500_000))]
ru_sents = pd.concat(ru_parts)["text"].sample(N_TRAIN, random_state=0).tolist()
uk_train = uk.loc[uk["is_valid"].eq(0), "text"].sample(N_TRAIN, random_state=0).tolist()
uk_valid = uk.loc[uk["is_valid"].eq(1), "text"].tolist()
{"test sentences": len(test_sents), "uk train": len(uk_train), "uk is_valid": len(uk_valid), "ru": len(ru_sents)}

{'test sentences': 2384, 'uk train': 30000, 'uk is_valid': 9913, 'ru': 30000}

In [3]:
class AdversarialRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.rnn = RNN(64, 64, batch_first=True, bidirectional=True)
        self.head = nn.Linear(128, 1)

    def forward(self, x):
        mask = (x > 0).float().unsqueeze(-1)
        output, _ = self.rnn(self.embedding(x))
        return self.head((output * mask).sum(1) / mask.sum(1).clamp(min=1)).squeeze(-1)


def adversarial_oof(positive, negative, epochs=EPOCHS):
    # Out-of-fold probability, 4 folds, that a text comes from `positive`.
    X = [[w.lower() for w in TOKEN.findall(x)][:MAX_TOKENS] or ["<empty>"] for x in positive + negative]
    y = np.r_[np.ones(len(positive)), np.zeros(len(negative))]
    oof = np.zeros(len(y))
    for fit_idx, hold_idx in StratifiedKFold(4, shuffle=True, random_state=0).split(X, y):
        counts = Counter(w for i in fit_idx for w in X[i])
        vocab = {w: i + 2 for i, (w, n) in enumerate(counts.most_common(30_000)) if n >= 2}
        encode = lambda idx: nn.utils.rnn.pad_sequence([torch.tensor([vocab.get(w, 1) for w in X[i]]) for i in idx], batch_first=True).to(DEVICE)
        model = AdversarialRNN(len(vocab) + 2).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), 2e-3)
        pos_weight = torch.tensor([float((y == 0).sum() / (y == 1).sum())], dtype=torch.float32, device=DEVICE)
        for _ in range(epochs):
            model.train()
            order = np.random.permutation(fit_idx)
            for start in range(0, len(order), 256):
                batch = order[start:start + 256]
                target = torch.tensor(y[batch], dtype=torch.float32, device=DEVICE)
                loss = nn.functional.binary_cross_entropy_with_logits(model(encode(batch)), target, pos_weight=pos_weight)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        model.eval()
        with torch.no_grad():
            for start in range(0, len(hold_idx), 1024):
                batch = hold_idx[start:start + 1024]
                oof[batch] = model(encode(batch)).cpu().numpy()
    return oof, y

In [4]:
strip_list = lambda xs: [strip_noise(x) for x in xs]
n_tokens = lambda xs: np.array([len(TOKEN.findall(x)) for x in xs])


def length_only_auc(positive, negative):
    auc = roc_auc_score(np.r_[np.ones(len(positive)), np.zeros(len(negative))], np.r_[n_tokens(positive), n_tokens(negative)])
    return max(auc, 1 - auc)


results, uk_scores = {}, None
for name, positive, negative in [
    ("test vs uk train, raw text", test_sents, uk_train),
    ("test vs uk train, no punctuation/emoji/digits", strip_list(test_sents), strip_list(uk_train)),
    ("test vs ru, no punctuation/emoji/digits", strip_list(test_sents), strip_list(ru_sents)),
    ("uk is_valid vs uk train, no punctuation/emoji/digits", strip_list(uk_valid), strip_list(uk_train)),
]:
    oof, y = adversarial_oof(positive, negative, epochs=2 if "is_valid" in name else EPOCHS)
    results[name] = {"BiGRU AUC": roc_auc_score(y, oof), "length-only AUC": length_only_auc(positive, negative)}
    if name.startswith("test vs uk train, no"):
        uk_scores, test_scores = oof[len(positive):], oof[:len(positive)]
pd.DataFrame(results).T.round(4)

,BiGRU AUC,length-only AUC
"test vs uk train, raw text",0.8698,0.6775
"test vs uk train, no punctuation/emoji/digits",0.8410,0.6908
"test vs ru, no punctuation/emoji/digits",0.9881,0.7264
"uk is_valid vs uk train, no punctuation/emoji/digits",0.4973,0.5045


In [5]:
# Which train sentences look like test? Candidates for a test-like validation slice or sample weights.
scores = pd.Series(uk_scores, index=range(len(uk_train)))
threshold = np.median(test_scores)
print(f"uk train sentences scoring above the test median: {(uk_scores > threshold).mean():.1%}")
display(pd.DataFrame({
    "test-like uk train": [uk_train[i] for i in scores.nlargest(10).index],
    "train-like test sentences": [test_sents[i] for i in np.argsort(test_scores)[:10]],
}))
pd.Series(uk_scores).describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_frame("uk train score")

uk train sentences scoring above the test median: 5.1%


,test-like uk train,train-like test sentences
0,"Режисер пояснив її так: “Це температура,.","Повний місяць також особливий тим, що це об’єднання Супермісяця та блакитного місяця (термін не має нічого спільного з кольором), пише Space."
1,"Встановлюються особи, причетні до вбивства"", - повідомили нам у поліції Київської області.",Також тут робив свої перші кроки в футболі єдиний в Києві олімпійський чемпіон з футболу 1988 року Олексій Михайличенко - ще зовсім малим він з батьками жив...
2,"Тож активісти для наших захисників ліплять вареники, печуть гречано-картопляні пироги та інші смаколики.","Це в 2,5 рази більше, ніж минулого року, коли лише 94 тисячі батьків вирішили відправити дітей на офлайн-навчання."
3,ДТП трапилася ввечері 2 вересня 2015 року.,"Щоб разом вчитись всьому новому: прогулянки з друзями, перші уроки, дивитись серіали у теплому ліжку та їсти смаколики в улюбленому кафе 🧡"", - йдеться в пов..."
4,Такий комплекс робіт проводять у 4—5 будинках кожного району.,Прийти та подивитися як він працює на нього краще у сонячну погоду.
5,"""5 канал"" транслює прес-конференція міністра освіти і науки Лілії Гриневич.","Перш за все, викликає питання сторінка, на якій опублікували оголошення, що отримало найбільшу кількість поширень."
6,У Києві на проспекті Академіка Заболотного вантажівка протаранила маршрутку.,Тепер вони використовують секрети лікування фахівців з передових країн.
7,"Люди стануть легковірними, прагнутимуть сенсацій.","Ціна, яку платить за своє вільне європейське майбутнє український народ, – надто висока."
8,"Основний товарний асортимент ХТЗ представлений малими тракторами потужністю 35 к/с, а також великими колісними і гусеничними тракторами потужністю 150-280 к/с.","Що означає, що цих свинособак дуже боляче покарали 💪 у різних областях їх недокраїни було дуже багато вибухів 👌"
9,Наразі територію пошкодження огороджено і спеціалісти КП «Київтеплоенерго» почали роботи з ліквідації аварії.,"Проєкт ""Створення рівних можливостей для сліпих і людей з вадами зору"" отримав високу міжнародну оцінку."


,uk train score
count,30000.00
mean,-4.28
std,2.91
min,-12.28
50%,-4.54
90%,-0.32
99%,3.16
max,6.88


## Findings

| Comparison | BiGRU AUC | Length-only AUC |
|---|---|---|
| test vs uk train, raw text | 0.870 | 0.678 |
| test vs uk train, without punctuation/emoji/digits | 0.841 | 0.691 |
| test vs ru, without punctuation/emoji/digits | 0.988 | 0.726 |
| uk `is_valid` vs uk train | 0.497 | 0.505 |

- **Train and test are clearly different domains.** The BiGRU separates them with AUC 0.87, and still 0.84 after punctuation, emoji and digits are removed. The shift is therefore in vocabulary and topics, and not only in Telegram formatting.
- **Length explains part of it.** Sentence length alone gives AUC 0.68 to 0.69, but the BiGRU is well above that. Test sentences here still contain channel signatures, which helps the classifier.
- **`is_valid` looks like train** (AUC 0.50). It is a random split from the same domain, so it can only show that the model trains and cannot predict leaderboard performance.
- **ru is the most distant source** (AUC 0.99), mostly because of language. Only 6.9% of test posts are in Russian, so ru should be added in limited amounts.
- **Only 5.1% of uk train sentences score above the test median.** The most test-like ones include Kyiv city news, such as accidents, police reports and municipal repairs, so a test-like slice exists, but it is small. It is a candidate for a harder validation slice or sample weights in experiments.

**What this means for validation.** Silver CV will overestimate leaderboard performance and correlate weakly with it, because of this shift and the different labelling policy. Use silver folds to check learning, the labelled sample as a sanity check, and submissions for decisions that depend on the domain.

**When the main model is chosen,** change `RNN` at the top and re-run, so that the check uses the same model as the task requires.